# Beyond Majority Voting: Selecting LLM Answers via Hidden State Trajectory Probes

**Author:** Nikolay Yudin (`n.yudin@gmail.com`)
**Repository:** [github.com/nick-yudin/Generalization/.../Latent\_control](https://github.com/nick-yudin/Generalization/tree/main/papers/Latent_control)

## Abstract

When a language model generates multiple candidate answers, how should we pick the best one? The default strategy — majority voting — treats the model as a black box, discarding everything except final answer strings. We show that the model's internal computations already contain a usable signal for answer quality, and that a remarkably simple method can extract it.

We propose *trajectory probes*: lightweight linear models trained on hidden-state features aggregated across the generation process. From each candidate answer, we extract mean, standard deviation, and final-token activations at eight evenly spaced layers, projected to 256 dimensions — a 6,144-dimensional trajectory fingerprint. A logistic regression probe trained with a pairwise ranking objective (RankNet) learns to prefer correct answers over incorrect ones from the same question.

On TriviaQA (Llama-3.1-8B-Instruct, *t*=0.3, *n*=500, *K*=4, 3 seeds), the probe improves over majority voting by **+5.1 ± 0.1 pp** (56.4% vs 51.3%), recovering 58% of the gap to the oracle upper bound, with a selection precision (PickAcc) of 91.2 ± 1.7%. On MATH (*n*=500, *K*\_gen=6, 3 seeds), the relationship is *K*-dependent: the probe outperforms MV by +2.1 pp at *K*\_eval=2 (all seeds positive) but the advantage narrows to +0.6 pp at *K*\_eval=4, as MV benefits more from additional votes in mathematical reasoning. The probe trains in under 60 seconds on CPU from 500 examples and adds zero latency at inference.

Two findings surprised us. First, the choice of training objective matters more than feature quality: a binary classifier with higher cross-validated AUC can underperform a pairwise probe with lower AUC, because ranking among candidates is a fundamentally different task than classifying correctness in isolation. Second, the per-layer signal distribution acts as a domain fingerprint — factual recall spreads information across all layers while mathematical reasoning concentrates it in the final third — yet a single probe trained on mixed-domain data matches domain-specific specialists with no interference.

Our results suggest that the "verifier" for best-of-*K* selection need not be a separate model or an additional LLM call. It can be a linear function of what the model already computes.

---
## Notebook 02: Key Ablations & Diagnostics

**Runtime:** CPU only (from pre-computed data), ~2 minutes.
**Input:** `data/paper2_ablation_summary.json`, `data/paper2_layer_aucs.json`, `data/paper2_test_ckpts.json`

These analyses defend the paper's claims against potential reviewer concerns.

### Sections
1. **Objective mismatch:** binary vs pairwise (high AUC ≠ good selection)
2. **Length confound:** with/without length features
3. **Per-layer AUC:** domain fingerprint (TriviaQA vs MATH)
4. **Cross-domain transfer:** train-domain × test-domain + merged probe
5. **Answer diversity:** breakdown by number of unique answers
6. **Margin calibration:** probe confidence → P(correct)

### Expected outputs
- 4–6 figures supporting the paper's analysis sections
- Printed statistics for each ablation

In [ ]:
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt

for p in ['.', '..', '/content/paper2_release', '/content/repo/papers/Latent_control']:
    if os.path.exists(os.path.join(p, 'paper2_utils.py')):
        sys.path.insert(0, p)
        break
    if os.path.exists(os.path.join(p, 'data')):
        DATA = os.path.join(p, 'data')

from paper2_utils import bootstrap_delta, mcnemar_test

# Load data
try:
    ablations = json.load(open(f'{DATA}/paper2_ablation_summary.json'))
    print(f'Loaded ablation summary: {list(ablations.keys())}')
except FileNotFoundError:
    ablations = {}
    print('WARNING: paper2_ablation_summary.json not found')

try:
    layer_aucs = json.load(open(f'{DATA}/paper2_layer_aucs.json'))
    print(f'Loaded layer AUCs: {list(layer_aucs.keys())}')
except FileNotFoundError:
    layer_aucs = {}

try:
    test_ckpts = json.load(open(f'{DATA}/paper2_test_ckpts.json'))
    print(f'Loaded test checkpoints: {list(test_ckpts.keys())}')
except FileNotFoundError:
    test_ckpts = {}

## 1. Objective Mismatch: Binary vs Pairwise

A binary classifier trained to predict correctness can achieve *higher* cross-validated AUC than a pairwise probe — yet perform *worse* at the actual selection task. This is because ranking among K candidates from the same question is fundamentally different from classifying each candidate independently.

In [ ]:
# From v10 (MATH) and v15 (TriviaQA): binary vs pairwise comparison
print('='*60)
print('OBJECTIVE MISMATCH: Binary AUC > Pairwise AUC, but Pairwise wins')
print('='*60)

# These are from the experimental results
comparisons = [
    {'domain': 'MATH (v10)', 'binary_auc': 0.887, 'pairwise_auc': 0.834,
     'binary_acc': 47.4, 'pairwise_acc': 49.0, 'mv_acc': 47.4},
    {'domain': 'TriviaQA (v15)', 'binary_auc': 0.954, 'pairwise_auc': 0.979,
     'binary_acc': 56.0, 'pairwise_acc': 60.4, 'mv_acc': 57.0},
]

print(f'\n{"Domain":<20} {"Bin AUC":>8} {"Pair AUC":>9} {"Bin Acc":>8} '
      f'{"Pair Acc":>9} {"MV":>6}')
print('-'*60)
for c in comparisons:
    print(f'{c["domain"]:<20} {c["binary_auc"]:>8.3f} {c["pairwise_auc"]:>9.3f} '
          f'{c["binary_acc"]:>7.1f}% {c["pairwise_acc"]:>8.1f}% '
          f'{c["mv_acc"]:>5.1f}%')

print('\n→ On MATH: Binary AUC (0.887) > Pairwise AUC (0.834)')
print('  But Pairwise accuracy (49.0%) > Binary accuracy (47.4%)')
print('  The objective must match the task: ranking, not classification.')

## 2. Length Confound

Does the probe simply learn that longer answers are better? We train probes with and without token-count features.

In [ ]:
print('='*60)
print('LENGTH CONFOUND ANALYSIS')
print('='*60)

if 'length_confound' in ablations:
    lc = ablations['length_confound']
    variants = [
        ('MV (baseline)', lc.get('mv_acc', 0)),
        ('Length-only probe', lc.get('acc_length_only', 0)),
        ('Binary (no length)', lc.get('acc_binary_nolen', lc.get('acc_binary', 0))),
        ('Pairwise (no length)', lc.get('acc_pairwise_nolen', lc.get('acc_pairwise', 0))),
        ('Binary (with length)', lc.get('acc_binary', 0)),
        ('Pairwise (with length)', lc.get('acc_pairwise', 0)),
    ]
    variants = [(n, v) for n, v in variants if v > 0]

    print(f'\n{"Variant":<25} {"Accuracy":>10}')
    print('-'*40)
    for name, acc in variants:
        print(f'{name:<25} {acc*100:>9.1f}%')

    if len(variants) >= 3:
        fig, ax = plt.subplots(figsize=(9, 4))
        names, accs = zip(*variants)
        colors = ['gray', 'khaki', 'lightblue', 'lightcoral', 'steelblue', 'coral']
        ax.bar(range(len(accs)), [a*100 for a in accs],
               color=colors[:len(accs)], edgecolor='white')
        ax.set_xticks(range(len(accs)))
        ax.set_xticklabels(names, rotation=15, ha='right')
        ax.set_ylabel('Accuracy (%)')
        ax.set_title('Length Confound — MATH (v12)')
        ax.grid(alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig('fig_length_confound.png', dpi=150, bbox_inches='tight')
        plt.show()

    print('\n→ Pairwise probe WITHOUT length features still beats MV.')
    print('  Length is a weak signal, not the primary one.')
else:
    print('No length confound data available.')

## 3. Per-Layer AUC: Domain Fingerprint

The distribution of probe signal across layers differs sharply by domain: distributed for factual recall (TriviaQA), concentrated in late layers for reasoning (MATH).

In [ ]:
print('='*60)
print('PER-LAYER PROBE AUC')
print('='*60)

if layer_aucs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, (domain, label, color) in zip(axes,
        [('trivia', 'TriviaQA', 'steelblue'), ('math', 'MATH', 'coral')]):
        if domain not in layer_aucs:
            ax.set_title(f'{label} (no data)')
            continue
        aucs = layer_aucs[domain]['layer_aucs']
        n = len(aucs)
        ax.bar(range(n), aucs, color=color, alpha=0.8, edgecolor='white')
        ax.axhline(0.5, color='gray', ls='--', lw=0.8, label='Random')
        ax.set_xlabel('Layer index')
        ax.set_title(label)
        ax.set_ylim(0.45, max(max(aucs), 0.7) * 1.05)

        # Mark spread8 layers
        spread8 = [int(round(i * (n - 1) / 7)) for i in range(8)]
        for l in spread8:
            ax.axvline(l, color='green', ls=':', alpha=0.3, lw=0.8)

        top_layers = sorted(range(n), key=lambda i: aucs[i], reverse=True)[:5]
        print(f'\n{label} top-5 layers: {top_layers}')
        print(f'  AUCs: {[f"{aucs[l]:.3f}" for l in top_layers]}')

    axes[0].set_ylabel('Pairwise probe AUC')
    plt.suptitle('Per-Layer Probe AUC — Domain Fingerprint', fontsize=13)
    plt.tight_layout()
    plt.savefig('fig_layer_auc.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No layer AUC data available.')

## 4. Cross-Domain Transfer

A probe trained on TriviaQA data tested on MATH (and vice versa). A merged probe trained on both matches domain-specific specialists.

In [ ]:
print('='*60)
print('CROSS-DOMAIN TRANSFER')
print('='*60)

if 'cross_domain' in ablations:
    cd = ablations['cross_domain']
    print('\nCross-domain results (v16b):')
    print(json.dumps({k: v for k, v in cd.items()
                      if k not in ['layer_aucs']}, indent=2, default=float)[:1000])

    print('\n→ Key finding: merged probe (trained on MATH+TriviaQA combined)')
    print('  matches domain-specific specialists with ≤0.0pp degradation.')
    print('  This suggests a shared latent quality signal across domains.')
else:
    print('No cross-domain data available.')

## 5. Answer Diversity

When all K answers are identical, MV always picks correctly (if the answer is correct). The probe's advantage appears when answers diverge.

In [ ]:
print('='*60)
print('ANSWER DIVERSITY ANALYSIS')
print('='*60)

# Analyze from test_ckpts: how many unique answers per question?
for name, ckpt in test_ckpts.items():
    if not ckpt or 'mv_correct' not in ckpt[0]:
        continue
    by_nunique = {}
    for q in ckpt:
        preds = [a.get('pred', '') or '' for a in q['attempts']]
        n_unique = len(set(p for p in preds if p))
        if n_unique == 0:
            n_unique = 1
        by_nunique.setdefault(n_unique, []).append(q)

    print(f'\n{name}:')
    print(f'  {"#unique":>7} {"Count":>6} {"MV acc":>8} {"Probe acc":>10} {"Δ":>8}')
    print('  ' + '-'*45)
    for nu in sorted(by_nunique):
        qs = by_nunique[nu]
        mv = sum(q['mv_correct'] for q in qs) / len(qs)
        pr = sum(q['probe_correct'] for q in qs) / len(qs)
        print(f'  {nu:>7} {len(qs):>6} {mv:>7.1%} {pr:>9.1%} '
              f'{(pr-mv)*100:>+7.1f}pp')

## 6. Margin Calibration

When the probe assigns a large margin (score gap between top-1 and top-2), is it more likely correct?

In [ ]:
print('='*60)
print('MARGIN CALIBRATION')
print('='*60)

for name, ckpt in test_ckpts.items():
    if not ckpt or len(ckpt) < 100 or 'probe_correct' not in ckpt[0]:
        continue
    margins, correct = [], []
    for q in ckpt:
        scores = [a.get('score_pairwise', 0) for a in q['attempts']]
        if len(scores) < 2:
            continue
        sorted_s = sorted(scores, reverse=True)
        margins.append(sorted_s[0] - sorted_s[1])
        correct.append(q['probe_correct'])

    if not margins:
        continue

    margins = np.array(margins)
    correct = np.array(correct)

    # Bin by margin quantile
    n_bins = 5
    edges = np.percentile(margins, np.linspace(0, 100, n_bins + 1))
    edges[0] -= 1e-9

    print(f'\n{name} ({len(margins)} questions):')
    print(f'  {"Margin range":>20} {"N":>5} {"Accuracy":>10} {"PickAcc":>10}')
    print('  ' + '-'*50)
    for i in range(n_bins):
        mask = (margins > edges[i]) & (margins <= edges[i+1])
        n = mask.sum()
        acc = correct[mask].mean() if n > 0 else 0
        print(f'  [{edges[i]:>7.2f}, {edges[i+1]:>7.2f}] {n:>5} '
              f'{acc:>9.1%}')

    # Plot
    fig, ax = plt.subplots(figsize=(6, 4))
    bin_centers = [(edges[i] + edges[i+1])/2 for i in range(n_bins)]
    bin_accs = []
    for i in range(n_bins):
        mask = (margins > edges[i]) & (margins <= edges[i+1])
        bin_accs.append(correct[mask].mean() if mask.sum() > 0 else 0)
    ax.bar(range(n_bins), [a*100 for a in bin_accs],
           color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_xticks(range(n_bins))
    ax.set_xticklabels([f'Q{i+1}' for i in range(n_bins)])
    ax.set_xlabel('Margin quintile (low → high)')
    ax.set_ylabel('Probe accuracy (%)')
    ax.set_title(f'Margin Calibration — {name}')
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'fig_margin_{name}.png', dpi=150, bbox_inches='tight')
    plt.show()

---

**Summary of ablation findings:**

1. **Objective mismatch** is real: pairwise > binary for selection, even when binary has higher AUC.
2. **Length is not a confound**: probe without length features still beats MV.
3. **Layer signal is domain-specific**: distributed (TriviaQA) vs late-concentrated (MATH).
4. **Cross-domain transfer works**: merged probe = specialist on both domains.
5. **Probe advantage grows with answer diversity** (more unique answers → bigger Δ).
6. **Margin calibration** is noisy but directional on TriviaQA; non-monotonic on MATH (needs more data).